# Group XX - Machine Learning, Element 1: Apply dataset preparation mechanisms
| Item | Details |
|---|---|
| Group | **Group XX** |
| Members | **IGIHOZO Bienvenue** and **MUGISHA Aime Patrick** |
| Question | **Q1 - Predicting maize yield for farmer cooperatives in Rwanda (MCSU)** |
| Stage | **4 - Clean and pre-process the data** |
| Driver / Navigator | Driver: MUGISHA Aime Patrick / Navigator: IGIHOZO Bienvenue |


## Stage 4 - Clean and pre-process the data (PC 1.3)
Every step calls a documented function in `src/maize_pipeline.py`. Each step shows a before/after view and writes to the
**Cleaning Log** - also the checks that found nothing.

In [1]:
import sys  # lets us tell Python where our own code (src/) is
from pathlib import Path  # file paths that work on every computer
import warnings  # hide harmless library notices
warnings.filterwarnings("ignore")  # keep the output readable
import numpy as np  # maths on whole columns
import pandas as pd  # the DataFrame table
ROOT = Path.cwd().parent  # notebooks/ is inside q1_maize_yield
sys.path.insert(0, str(ROOT / "src"))  # make src/maize_pipeline.py importable
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)  # show every column
RAW = ROOT / "data" / "raw" / "maize_yield_rwanda_rw.csv"  # the original file (never edited)
import maize_pipeline as mp  # our reusable functions

raw = pd.read_csv(RAW)  # original file
log = mp.CleaningLog()  # empty log
def before_after(before, after, cols):  # helper: shape, unique values, missing counts
    print(f"shape: {before.shape} -> {after.shape}")  # size
    for c in cols:  # each column
        print(f"  {c:<22} unique {before[c].nunique() if c in before else '-'} -> {after[c].nunique() if c in after else '-'}"
              f" | missing {before[c].isna().sum() if c in before else '-'} -> {after[c].isna().sum() if c in after else '-'}")  # one line

### Steps 1-2: spaces and duplicates

In [2]:
df = mp.trim_text(raw, log)  # remove spaces around labels
step = df.copy()  # before duplicates
df = mp.check_duplicates(df, log)  # same id / same content
before_after(step, df, ["record_id"])  # no change expected

shape: (1426, 20) -> (1426, 20)
  record_id              unique 1426 -> 1426 | missing 0 -> 0


### Step 3: categories, district-province, official names

In [3]:
step = df.copy()  # before
df = mp.check_categories(df, log)  # allowed labels; province from district
before_after(step, df, ["province", "district", "season", "seed_variety"])  # province labels
print(step["province"].value_counts().to_dict(), "->", df["province"].value_counts().to_dict())  # 'Kigali' -> 'Kigali City'

shape: (1426, 20) -> (1426, 20)
  province               unique 5 -> 5 | missing 0 -> 0
  district               unique 30 -> 30 | missing 0 -> 0
  season                 unique 3 -> 3 | missing 0 -> 0
  seed_variety           unique 6 -> 6 | missing 0 -> 0
{'Southern': 388, 'Western': 348, 'Eastern': 328, 'Northern': 230, 'Kigali': 132} -> {'Southern': 388, 'Western': 348, 'Eastern': 328, 'Northern': 230, 'Kigali City': 132}


### Step 4: realistic ranges

In [4]:
step = df.copy()  # before
df = mp.check_ranges(df, log)  # out-of-range -> NaN
pd.concat({"before": step[list(mp.RANGES)].agg(["min", "max"]).T, "after": df[list(mp.RANGES)].agg(["min", "max"]).T}, axis=1)  # ranges

before                 after           
                            min        max        min        max
year                   2015.000   2024.000   2015.000   2024.000
altitude_m             1243.000   2341.000   1243.000   2341.000
rainfall_mm             150.000    933.300    150.000    933.300
avg_temp_c               15.400     23.000     15.400     23.000
soil_ph                   4.500      7.200      4.500      7.200
npk_kg_ha                 0.000    208.100      0.000    208.100
urea_kg_ha                0.000    135.200      0.000    135.200
manure_t_ha               0.000      6.860      0.000      6.860
plant_density_per_ha  19337.000  72248.000  19337.000  72248.000
farm_area_ha              0.100      4.220      0.100      4.220
pest_incidence_index      0.001      0.929      0.001      0.929
yield_t_ha                1.010      5.150      1.010      5.150

### Steps 5-6: missing-value indicator, target

In [5]:
step = df.copy()  # before
df = mp.flag_missing(df, log)  # indicator field_data_missing
df = mp.check_target(df, log)  # valid target + yield_kg_ha
before_after(step, df, mp.SOIL_TEST_COLS + ["field_data_missing", "yield_kg_ha"])  # new columns
df[df["field_data_missing"] == 1].head()  # farms with gaps

shape: (1426, 20) -> (1426, 22)
  soil_ph                unique 211 -> 211 | missing 14 -> 14
  pest_incidence_index   unique 563 -> 563 | missing 14 -> 14
  rainfall_mm            unique 1225 -> 1225 | missing 11 -> 11
  field_data_missing     unique - -> 2 | missing - -> 0
  yield_kg_ha            unique - -> 308 | missing - -> 0


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha,field_data_missing,yield_kg_ha
0,RW-MZ-00001,2015.0,A,Eastern,Bugesera,1349.0,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803.0,0.63,No,NaN,2.71,1,2710.0
38,RW-MZ-00039,2015.0,A,Southern,Nyamagabe,1934.0,715.1,16.6,NaN,Loam,RHM104,Traditional,26.6,0.0,0.00,37926.0,0.59,No,NaN,2.65,1,2650.0
257,RW-MZ-00258,2016.0,B,Western,Rutsiro,1925.0,NaN,19.1,NaN,Clay,ZM607,Improved,86.8,36.0,4.11,37764.0,1.08,No,NaN,2.43,1,2430.0
354,RW-MZ-00355,2017.0,B,Southern,Kamonyi,1551.0,370.6,20.5,NaN,Clay loam,Local,Traditional,30.9,27.3,0.00,38184.0,0.35,No,NaN,1.44,1,1440.0
418,RW-MZ-00419,2017.0,C,Kigali City,Nyarugenge,1504.0,NaN,20.5,NaN,Sandy loam,RHM104,Improved,114.5,71.5,1.13,55813.0,0.60,No,NaN,2.21,1,2210.0


### Reproducibility: the one-call pipeline gives exactly the same table

In [6]:
again, _ = mp.clean_maize(pd.read_csv(RAW))  # all steps in one call
pd.testing.assert_frame_equal(df, again)  # error if anything differs
print("identical:", again.shape)  # confirmation

identical: (1426, 22)


### Cleaning Log - saved to `docs/cleaning_log.csv`

In [7]:
clean_log = log.to_frame()  # every decision
clean_log.to_csv(ROOT / "docs" / "cleaning_log.csv", index=False)  # save
(ROOT / "data" / "interim").mkdir(parents=True, exist_ok=True)  # folder
df.to_csv(ROOT / "data" / "interim" / "maize_clean.csv", index=False)  # clean dataset
clean_log[["step", "operation", "column", "problem", "action", "records_affected"]]  # display

,step,operation,column,problem,action,records_affected
0,1,trim_text,all text columns,leading/trailing spaces,stripped (check),0
1,2,duplicates,record_id,same record_id repeated,removed (keep first),0
2,3,duplicates,all except record_id,same content under a different record_id,removed (keep first),0
3,4,categories,season,labels outside the allowed list,check only,0
4,5,categories,soil_type,labels outside the allowed list,check only,0
5,6,categories,seed_variety,labels outside the allowed list,check only,0
6,7,categories,farming_practice,labels outside the allowed list,check only,0
7,8,categories,irrigation,labels outside the allowed list,check only,0
8,9,categories,province,'Kigali' instead of the official 'Kigali City',renamed,132
9,10,categories,"district, province",district written under the wrong province,province taken from district,0


### Split: 70 / 15 / 15, seed 42, stratified by season - **before** imputing and scaling
Imputers and scalers learn numbers (medians, means, standard deviations); if they saw validation/test rows, information from
those rows would leak into training. Stratifying by season keeps the small Season C group in all three files.

In [8]:
from sklearn.model_selection import train_test_split  # random split
train, temp = train_test_split(df, test_size=0.30, random_state=mp.SEED, stratify=df["season"])  # 70 % train
val, test = train_test_split(temp, test_size=0.50, random_state=mp.SEED, stratify=temp["season"])  # 15 % + 15 %
thr = mp.low_yield_threshold(train)  # bottom 10 % learned on TRAIN only
train, val, test = (mp.add_low_yield_flag(p, thr) for p in (train, val, test))  # same threshold everywhere
print(f"low-yield threshold (train 10th percentile): {thr:.3f} t/ha = {thr * 1000:.0f} kg/ha")  # value
for n, p in [("train", train), ("val", val), ("test", test)]:  # each split
    print(f"{n:<5} {p.shape}  season C {int((p.season == 'C').sum()):>3}  low-yield share {p.low_yield_risk.mean():.3f}")  # size + balance
    p.to_csv(ROOT / "data" / "interim" / f"{n}_clean.csv", index=False)  # save

low-yield threshold (train 10th percentile): 1.667 t/ha = 1667 kg/ha
train (998, 23)  season C 158  low-yield share 0.100
val   (214, 23)  season C  34  low-yield share 0.089
test  (214, 23)  season C  34  low-yield share 0.103


**Imbalance:** `low_yield_risk` is about 10 % positives by construction -> planned remedy for the modelling team:
`class_weight="balanced"` or resampling **on the training set only**.

### Imputation, encoding, scaling - one scikit-learn Pipeline fitted on train only

In [9]:
from sklearn.compose import ColumnTransformer  # different steps for different columns
from sklearn.pipeline import Pipeline  # chain of steps
from sklearn.impute import SimpleImputer  # fills missing values
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # encoding and scaling
numeric_cols = ["altitude_m", "rainfall_mm", "avg_temp_c", "soil_ph", "npk_kg_ha", "urea_kg_ha", "manure_t_ha",
                "plant_density_per_ha", "farm_area_ha", "pest_incidence_index", "year"]  # numbers
nominal_cols = ["season", "district", "soil_type", "seed_variety", "farming_practice", "irrigation"]  # categories
base_prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),  # median: robust to the kept outliers
                      ("scale", StandardScaler())]), numeric_cols),  # mean 0, std 1
    ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_cols),  # one 0/1 column per label
])
base_prep.fit(train)  # LEARN medians, means, stds and labels from TRAIN only
Xtr, Xva, Xte = (base_prep.transform(p) for p in (train, val, test))  # APPLY to all three
print("train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)  # same columns
pd.DataFrame(Xtr, columns=base_prep.get_feature_names_out()).iloc[:5, :12].round(2)  # first rows

train (998, 59) | val (214, 59) | test (214, 59)


,num__altitude_m,num__rainfall_mm,num__avg_temp_c,num__soil_ph,num__npk_kg_ha,num__urea_kg_ha,num__manure_t_ha,num__plant_density_per_ha,num__farm_area_ha,num__pest_incidence_index,num__year,nom__season_A
0,-1.06,-0.88,1.20,1.05,-0.21,-0.48,1.37,-0.92,-0.77,-1.33,-0.53,0.0
1,1.30,0.42,-1.83,-0.28,-1.00,-0.90,0.70,-1.17,-0.45,0.05,0.53,0.0
2,-0.22,1.03,-0.80,-0.37,0.03,-0.32,-0.56,0.79,1.13,1.70,0.18,0.0
3,-0.17,0.23,0.23,1.63,0.52,-0.76,-1.25,0.25,1.84,1.23,1.23,0.0
4,-1.66,-0.23,0.41,0.60,0.76,0.20,0.47,0.59,1.37,-0.99,1.58,0.0


In [10]:
pd.DataFrame(Xtr[:, :len(numeric_cols)], columns=numeric_cols).agg(["mean", "std"]).round(2)  # ~0 and ~1 = scaler worked

,altitude_m,rainfall_mm,avg_temp_c,soil_ph,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,pest_incidence_index,year
mean,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


**Leakage question:** "Why fit the imputer and scaler on train only?" - validation and test stand for future farms; their
medians, means and standard deviations must not influence anything the model learns.